# INFORME EDA (Exploratory Data Analysis) Y FASE ANALÍTICA 1
Este estudio exhaustivo aborda el dataset de OWID desde 0, ignorando preconcepciones para encontrar el peso real de cada métrica.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
warnings = __import__('warnings')
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## PARTE I: Ingesta y Diagnóstico Exploratorio Básico
Inspeccionamos peso, tipos de datos y matriz de nulos (Missingness).

In [ ]:
url = 'https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv'
raw_df = pd.read_csv(url)
print(f'Dimensiones Iniciales: {raw_df.shape}')
display(raw_df.info())

### Variables Financieras y Demográficas Disponibles
Aislamos columnas clave para ver qué podemos correlacionar (gdp, population, primary_energy_consumption, y todos los gases: co2, methane, nitrous_oxide).

In [ ]:
cols = ['country', 'year', 'iso_code', 'population', 'gdp', 'co2', 'methane', 'nitrous_oxide', 'primary_energy_consumption', 'co2_per_capita']
df_core = raw_df[cols].copy()
display(df_core.head())

### Matriz de Valores Faltantes por Década
Identificamos a partir de qué era moderna tenemos datos útiles y robustos.

In [ ]:
df_core['decade'] = (df_core['year'] // 10) * 10
missing_by_decade = df_core.groupby('decade').apply(lambda x: x.isnull().mean() * 100)
plt.figure(figsize=(12, 6))
sns.heatmap(missing_by_decade[['population', 'gdp', 'co2', 'methane']], cmap='YlOrRd', cbar_kws={'label': '% de Datos Nulos'})
plt.title('Disponibilidad de Datos (Porcentaje de Nulos) a través del Tiempo')
plt.show()

## PARTE II: Purgado de Datos (ISO Standard)
Retiramos los continentes y bloques globales para aislar naciones viables.

In [ ]:
df_countries = df_core[df_core['iso_code'].notna() & ~df_core['iso_code'].str.startswith('OWID', na=False)].copy()
print(f'Naciones Soberanas Viables aisladas: {df_countries.country.nunique()} países.')
display(df_countries['country'].sample(10).values)

### Distribución de Emisiones CO2 a Nivel Global (Sin Nulos Temporales)
Buscamos Outliers. ¿Qué tipo de distribución estadística sigue el carbono?

In [ ]:
modern_df = df_countries[df_countries['year'] == 2022] # Usamos un año muy reciente y completo
plt.figure(figsize=(10, 4))
sns.histplot(modern_df['co2'].dropna(), bins=50, kde=True)
plt.title('Distribución Absoluta de C02 en Naciones Soberanas (2022)')
plt.xlabel('Millones de Toneladas (CO2)')
plt.yscale('log')
plt.show()
print("Observación: Distribución fuertemente asimétrica a la derecha. Pocos países emiten casi todo el carbono.")

## PARTE III: Matriz de Correlaciones Univariables
¿Es la población o el PIB el principal detonante del CO2? Usamos correlación de Spearman para eludir Outliers.

In [ ]:
corrmat = modern_df[['population', 'gdp', 'co2', 'methane', 'primary_energy_consumption']].corr(method='spearman')
plt.figure(figsize=(8, 6))
sns.heatmap(corrmat, annot=True, cmap='coolwarm', vmin=0, vmax=1)
plt.title('Matriz de Correlación de Spearman (Año Base: 2022)')
plt.show()
display(corrmat)

### El Factor 'Methane' y otros GEI olvidados
Verificamos la proporción de emisión de metano frente a CO2 puro.

In [ ]:
df_gei = df_countries.groupby('year')[['co2', 'methane', 'nitrous_oxide']].sum().replace(0, np.nan).dropna()
plt.figure(figsize=(10, 5))
plt.plot(df_gei.index, df_gei['co2'], label='CO2 (M Toneladas)')
plt.plot(df_gei.index, df_gei['methane'], label='Methane (M Toneladas equivalentes)')
plt.plot(df_gei.index, df_gei['nitrous_oxide'], label='Nitrous Oxide')
plt.title('Evolución Agregada de Gases de Efecto Invernadero')
plt.legend()
plt.show()

## PARTE IV: Diagnóstico Analítico y Desigualdad
Si la riqueza es la que quema carbono: dividimos el mundo en Cuartiles de Riqueza (PIB Per Cápita).

In [ ]:
modern_df['gdp_pc'] = modern_df['gdp'] / modern_df['population']
quartiles = pd.qcut(modern_df['gdp_pc'].dropna(), q=4, labels=['Bajo (Q1)', 'Medio-Bajo (Q2)', 'Medio-Alto (Q3)', 'Alto (Q4)'])
modern_df['Riqueza'] = quartiles

riqueza_stats = modern_df.groupby('Riqueza')[['co2', 'population']].sum()
riqueza_stats['CO2_share_%'] = (riqueza_stats['co2'] / riqueza_stats['co2'].sum()) * 100
riqueza_stats['Pop_share_%'] = (riqueza_stats['population'] / riqueza_stats['population'].sum()) * 100
display(riqueza_stats)

### Visualizando la Injusticia Gini del CO2
¿El 25% más rico emite cuánto del total?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(riqueza_stats['CO2_share_%'], labels=riqueza_stats.index, autopct='%1.1f%%', colors=sns.color_palette('Reds'))
axes[0].set_title('Proporción de Emisión CO2 por Nivel de Riqueza')
axes[1].pie(riqueza_stats['Pop_share_%'], labels=riqueza_stats.index, autopct='%1.1f%%', colors=sns.color_palette('Blues'))
axes[1].set_title('Proporción Demográfica Mundial')
plt.show()

## PARTE V: Top 10 Movilidad y Transición Hegemónica
Identificamos las matrices líderes de emisión en cortes de 50 años (1920, 1970, 2022).

In [ ]:
years = [1920, 1970, 2022]
for y in years:
    top = df_countries[df_countries['year'] == y].nlargest(5, 'co2')[['country', 'co2']]
    print(f'\n--- TOP 5 EN EL AÑO {y} ---')
    display(top)

## Conclusión de Extracción 1
El informe ha detectado las correlaciones reales, ha verificado el metano, ha segmentado la desigualdad por deciles económicos y revelado la mutación del Top 5 geográfico.